# Extra baselines — Sven vs the full optimizer suite

Reviewer-requested modern optimizers on the headline datasets (full data). Best final loss per method; does the wider comparison change the story?

> Loads the fresh Gram-backend results. Robust to partial data (plots whatever has finished).

In [ ]:
import sys, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, '.')
from style import load_results, average_over_seeds, set_style
from analysis_helpers import add_derived, best_per_method, valid, loss_curve, steps_to_target, method_order
from pathlib import Path
set_style()
PLOT_DIR = Path('plots/baselines'); PLOT_DIR.mkdir(parents=True, exist_ok=True)
def sven_color(m): return 'k' if m=='Sven' else None
def sven_lw(m):    return 2.6 if m=='Sven' else 1.6

In [ ]:
SCANS = {'toy_1d':'rebuttal_baselines_toy_1d_scan','polynomial':'rebuttal_baselines_polynomial_scan',
         'mnist':'rebuttal_baselines_mnist_scan'}
data={}
for tag,name in SCANS.items():
    try: data[tag]=add_derived(load_results(name))
    except FileNotFoundError: print('missing',name)

### Best final loss per optimizer

In [ ]:
metric = 'final_train_loss'
fig,axes=plt.subplots(1,len(data),figsize=(6*len(data),4.2),squeeze=False)
for ax,(tag,df) in zip(axes[0],data.items()):
    b=best_per_method(df, by=metric).sort_values(metric)
    colors=['crimson' if m=='Sven' else 'steelblue' for m in b['method']]
    ax.barh(b['method'], b[metric], color=colors); ax.set_xscale('log')
    ax.invert_yaxis(); ax.set_title(tag); ax.set_xlabel('best '+metric)
plt.tight_layout(); plt.savefig(PLOT_DIR/'best_loss_by_optimizer.pdf',bbox_inches='tight'); plt.show()

### Table

In [ ]:
for tag,df in data.items():
    b=best_per_method(df, by='final_train_loss').sort_values('final_train_loss')
    print(f'== {tag} ==')
    for _,r in b.iterrows(): print(f'  {r.method:10s} train={r.final_train_loss:.3e}  val={r.final_val_loss:.3e}')